
# Handwritten Digit Recognition using an Artificial Neural Network

This notebook documents the reproducible training and evaluation workflow used by the portfolio project. The packaged model is a tuned dense ANN trained on MNIST and recorded **98.24% test accuracy**.

The production Streamlit application uses the same saved model and the reusable modules under `src/`.


In [ ]:

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import prepare_mnist
from src.model_training import DEFAULT_PARAMS, build_ann, train_model
from src.model_evaluation import evaluate_model
from src.image_preprocessing import preprocess_digit_image
from src.prediction_pipeline import load_digit_model, predict_digit

MODEL_PATH = PROJECT_ROOT / "models" / "digit_recognition_model.keras"
OUTPUT_DIR = PROJECT_ROOT / "outputs"


## 1. Load and prepare MNIST

In [ ]:

data = prepare_mnist()
print("Training:", data.x_train.shape)
print("Validation:", data.x_validation.shape)
print("Test:", data.x_test.shape)
print("Pixel range:", data.x_train.min(), "to", data.x_train.max())


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for axis, image, label in zip(axes.flat, data.test_images[:10], data.y_test[:10]):
    axis.imshow(image, cmap="gray")
    axis.set_title(f"Label: {label}")
    axis.axis("off")
plt.tight_layout()


## 2. Inspect the tuned architecture

In [ ]:

model = build_ann(**{key: value for key, value in DEFAULT_PARAMS.items() if key != "batch_size"})
model.summary()
DEFAULT_PARAMS



## 3. Optional retraining

The model artifact is already included. Set `RUN_TRAINING = True` only when you want to retrain it. The training function builds a fresh final model after hyperparameter selection.


In [ ]:

RUN_TRAINING = False

if RUN_TRAINING:
    trained_model, history, selected_params = train_model(
        output_dir=PROJECT_ROOT / "models",
        epochs=30,
        tune=False,
    )
    print(selected_params)


## 4. Load the packaged model and evaluate

In [ ]:

saved_model = load_digit_model(MODEL_PATH)
saved_model.summary()


In [ ]:

RUN_EVALUATION = False

if RUN_EVALUATION:
    metrics = evaluate_model(
        model_path=MODEL_PATH,
        output_dir=OUTPUT_DIR,
        history_path=OUTPUT_DIR / "training_history.json",
    )
else:
    metrics = json.loads((OUTPUT_DIR / "model_metrics.json").read_text())
metrics


## 5. Inspect class-level performance

In [ ]:

pd.read_csv(OUTPUT_DIR / "per_class_accuracy.csv")


In [ ]:

from IPython.display import display, Image as NotebookImage

display(NotebookImage(filename=str(OUTPUT_DIR / "confusion_matrix.png")))
display(NotebookImage(filename=str(OUTPUT_DIR / "misclassified_digits.png")))


## 6. Run the production preprocessing and prediction pipeline

In [ ]:

sample_path = PROJECT_ROOT / "data" / "sample_digits" / "digit_7.png"
processed = preprocess_digit_image(sample_path)
prediction = predict_digit(saved_model, processed.model_input)

print("Predicted digit:", prediction.predicted_digit)
print("Confidence:", f"{prediction.confidence:.2%}")
print("Runner-up:", prediction.second_digit, f"({prediction.second_confidence:.2%})")

plt.imshow(processed.centered_28x28, cmap="gray")
plt.title("28×28 model input")
plt.axis("off")



## Summary

The project preserves the tuned ANN while adding production-oriented preprocessing, modular training and evaluation code, explicit limitations, automated tests, CI, and a Streamlit deployment path.
